# Example of TwoTower compiled inference

This example continues the [simple TwoTower example](simple_example.ipynb). The model and the data pipeline are the same; this notebook shows what happens after training: packaging the model for online serving with the `replay.nn.compiled` module.

The module exports a model's `forward_inference` to *ONNX* and compiles it with *OpenVINO*:

- **Two serving artifacts.** A TwoTower model consists of a query tower and an item tower. Online scoring only needs the query tower as a graph, plus an **item-embedding matrix** that `export_item_embeddings` computes once over the whole catalog. The item tower itself does not run online.
- **A self-sufficient checkpoint.** `capture_config` records the constructor parameters and the tensor schema in a sidecar file next to the checkpoint, so `restore_from_checkpoint` can rebuild the model without re-running the training code.
- **NumPy-only request encoding.** `encode_inputs` turns raw user events into model inputs, and `CompiledModel.predict` runs the compiled graph. The serving side needs neither PyTorch nor RePlay transforms: `load` reads the artifact with *OpenVINO* alone.

This example additionally requires the optional `onnx` and `openvino` packages (see the installation instructions in README).

In [1]:
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env ARROW_NUM_THREADS=8
%env ARROW_IO_THREADS=8

env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: ARROW_NUM_THREADS=8
env: ARROW_IO_THREADS=8


In [2]:
import lightning as L
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

L.seed_everything(42);

Seed set to 42


## Preparing data

In this example, we use the MovieLens dataset, namely the 1M subset. As in the simple example, only item ids are used as model input.

Note: the current implementation of `TensorSchema` in RePlay does not support `numpy` nullable dtypes, so we cast the timestamp column to `int64` in the next cells.

In [3]:
interactions = pd.read_csv("../data/ml1m_ratings.dat", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
interactions = interactions.drop(columns=["rating"])

item_features = pd.read_csv("../data/ml1m_items.dat", sep="\t", names=["item_id", "title", "genres"])
item_features = item_features.drop(columns=["title", "genres"])

In [4]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions

,user_id,item_id,timestamp
1000138,6040,858,0
999873,6040,593,1
1000153,6040,2384,2
1000192,6040,2019,3
1000007,6040,1961,4
...,...,...,...
825793,4958,2399,446
825438,4958,1407,447
825724,4958,3264,448
825731,4958,2634,449


### Encode categorical data

To ensure all categorical data is fit for training, it needs to be encoded using the `LabelEncoder` class.

We fit the encoder for the `item_id` column using the item features, because some items may be missing from interactions. The `user_id` column is fitted using interactions.

In [5]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)
encoder.rules[0].fit(interactions)
encoder.rules[1].fit(item_features)
encoded_interactions = encoder.transform(interactions)
encoded_interactions

,timestamp,user_id,item_id
0,0,6039,847
1,1,6039,589
2,2,6039,2315
3,3,6039,1950
4,4,6039,1892
...,...,...,...
1000204,446,4957,2330
1000205,447,4957,1384
1000206,448,4957,3195
1000207,449,4957,2565


In [6]:
item_features_encoded = encoder.rules[1].transform(item_features)
item_features_encoded

,item_id
0,0
1,1
2,2
3,3
4,4
...,...
3878,3878
3879,3879
3880,3880
3881,3881


### Split interactions and prepare datasets

We use the last-one-out splitting strategy and filter out cold users and items for simplicity.

`TwoTower` expects each user in the batch to provide their events as a sequence, so the splits are processed with the `groupby_sequences` utility. The ground truth is joined to the validation events to provide targets for the validation metrics.

In [7]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

validation_events, validation_gt = splitter.split(encoded_interactions)
train_events = validation_events

In [8]:
from replay.data.nn.utils import groupby_sequences


def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")


def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})
    return events_df.merge(gt_to_join, on="user_id", how="inner")


train_events = bake_data(train_events)
validation_events = add_gt_to_events(bake_data(validation_events), bake_data(validation_gt))
train_events

,user_id,timestamp,item_id
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3117, 1009, 1250, 1672, 2271, 1768, 3339, 118..."
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1180, 1199, 1192, 2648, 1273, 2874, 1207, 117..."
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[589, 2789, 3465, 1899, 1892, 1407, 1246, 1358..."
3,3,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1192, 1081, 3399, 476, 3458, 257, 1180, 1178,..."
4,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[2648, 896, 907, 352, 1230, 2119, 1111, 2789, ..."
...,...,...,...
6035,6035,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1672, 3369, 2359, 1814, 2307, 2758, 2614, 263..."
6036,6036,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[693, 1813, 1247, 1959, 3439, 558, 3079, 847, ..."
6037,6037,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3327, 908, 1192, 2077, 1366, 352, 1063, 1132,..."
6038,6038,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[109, 279, 1998, 1211, 918, 3064, 2953, 3019, ..."


In [9]:
from pathlib import Path

data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
PATH_ENCODED_FEATURES = data_dir / "item_features_encoded.parquet"
ENCODER_PATH = data_dir / "encoder"

In [10]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
item_features_encoded[["item_id"]].to_parquet(PATH_ENCODED_FEATURES)

encoder.save(ENCODER_PATH)

/home/coder/sdk/RePlay/replay/preprocessing/label_encoder.py:1002: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/home/coder/sdk/RePlay/replay/preprocessing/label_encoder.py:563: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


# Prepare for model training

### Create the tensor schema

A schema shows the correspondence of columns from the source dataset to the internal representation of tensors inside the model. It is required by the NN models to correctly create embeddings for every source column. Note that `user_id` is not required in `TensorSchema`.

Note that **cardinality** is the number of unique values in the item catalog (vocabulary). The **padding value** is the next value after the last one.

In [11]:
from replay.data import FeatureHint, FeatureSource, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorFeatureSource, TensorSchema

EMBEDDING_DIM = 64
NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
            feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")],
        )
    ]
)

### Configure `ParquetModule` and transformation pipelines

The `ParquetModule` class enables training of models on large datasets by reading data in a batch-wise way. Metadata describes the shape and the padding value of every input column.

In [12]:
from replay.nn.transform.template import make_default_twotower_transforms

MAX_SEQ_LEN = 50

transforms = make_default_twotower_transforms(tensor_schema)

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": tensor_schema["item_id"].padding_value},
    },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
        "ground_truth": {"shape": 1, "padding": -1},
    },
}

In [13]:
from replay.data.nn import ParquetModule

BATCH_SIZE = 32

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

/tmp/.ctx-mode-EzGjWj/ipykernel_45308/1761019238.py:5: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


## Train the model

### Build the model with `capture_config`

Instead of calling `TwoTower.from_params` directly, we build the model through `capture_config`. It returns the same model instance plus a `ModelConfig` that records the constructor parameters and the tensor schema — the recipe needed to rebuild the model from a checkpoint later.

The item tower reads the item catalog with a `FeaturesReader`; the catalog itself is cached in the model buffers during training and saved into the checkpoint, so the reader only rebuilds the module structure at restore time.

In [14]:
from replay.nn.compiled import capture_config
from replay.nn.sequential import TwoTower
from replay.nn.sequential.twotower import FeaturesReader

NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3

item_features_reader = FeaturesReader(
    schema=tensor_schema,
    metadata={"item_id": {}},
    path=PATH_ENCODED_FEATURES,
)

twotower, model_config = capture_config(
    TwoTower,
    tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    item_features_reader=item_features_reader,
)

In [15]:
from replay.nn.lightning import LightningModule

model = LightningModule(twotower)

### Run the training stage

We add the following callbacks:

1. `ModelCheckpoint` — saves the best trained model based on its Recall metric.
1. `ComputeMetricsCallback` — calculates validation metrics after every epoch.

Training is limited to two epochs: this example is about serving, not accuracy.

In [16]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from replay.nn.lightning.callback import ComputeMetricsCallback

checkpoint_callback = ModelCheckpoint(
    dirpath="twotower/checkpoints/",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall"],
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
    verbose=False,
)

csv_logger = CSVLogger(save_dir="twotower/.logs/train", name="TwoTower-compiled-example")

trainer = L.Trainer(
    max_epochs=2,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=csv_logger,
)

trainer.fit(model, datamodule=parquet_module)

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/home/coder/venvs/replay-onnx/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/coder/sdk/RePlay/examples/twotower/twotower/checkpoints exists and is not empty.


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ TwoTower │  352 K │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 352 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 352 K                                                                                                
Total estimated model params size (MB): 1.409                                                                      
Modules in train mode: 53                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

... (2 outputs left out)

/home/coder/venvs/replay-onnx/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your 
`IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), 
`__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.

Epoch 0, global step 189: 'recall@10' reached 0.03080 (best 0.03080), saving model to '/home/coder/sdk/RePlay/examples/twotower/twotower/checkpoints/epoch=0-step=189.ckpt' as top 1


Epoch 1, global step 378: 'recall@10' reached 0.10815 (best 0.10815), saving model to '/home/coder/sdk/RePlay/examples/twotower/twotower/checkpoints/epoch=1-step=378-v2.ckpt' as top 1


`Trainer.fit` stopped: `max_epochs=2` reached.


In [17]:
best_model_path = checkpoint_callback.best_model_path
best_model_path

'/home/coder/sdk/RePlay/examples/twotower/twotower/checkpoints/epoch=1-step=378-v2.ckpt'

## Make the checkpoint self-sufficient

`save_config` writes the captured `ModelConfig` next to the checkpoint. Together with the checkpoint file, this sidecar is everything `restore_from_checkpoint` needs to rebuild the model and its schema — without the training code, the schema definition, or the constructor parameters.

Note that the `item_features_reader` is deliberately not serialized into the sidecar; it is passed explicitly at restore time.

In [18]:
from replay.nn.compiled import read_config, save_config

config_path = save_config(model_config, best_model_path)
print(config_path)

restored_config = read_config(best_model_path)
restored_config.model_type, restored_config.from_params

/home/coder/sdk/RePlay/examples/twotower/twotower/checkpoints/epoch=1-step=378-v2.ckpt.config.json


('twotower_query',
 {'embedding_dim': 64,
  'max_sequence_length': 50,
  'num_heads': 2,
  'num_blocks': 2,
  'dropout': 0.3})

## Restore the model from the checkpoint

`restore_from_checkpoint` rebuilds the model and the tensor schema from the checkpoint plus its config sidecar. For a `TwoTower` model, the `item_features_reader` argument is required to rebuild the item tower structure; the catalog tensors themselves come from the checkpoint.

In [19]:
from replay.nn.compiled import restore_from_checkpoint

twotower, schema = restore_from_checkpoint(best_model_path, item_features_reader=item_features_reader)
twotower.eval();

## Export the item-embedding matrix

Online scoring multiplies query embeddings by an item-embedding matrix. `export_item_embeddings` runs the item tower once over the whole catalog and returns an `ItemEmbeddings` artifact with the `[num_items, embedding_dim]` matrix.

From this point on, the item tower is not needed: the matrix is the second serving artifact.

In [20]:
from replay.nn.compiled import export_item_embeddings

item_embeddings = export_item_embeddings(twotower)
item_embeddings.matrix.shape, item_embeddings.num_items, item_embeddings.embedding_dim

((3883, 64), 3883, 64)

## Compile the query tower

`compile` exports the query tower's `forward_inference` to *ONNX* and compiles it with *OpenVINO*. The `EmbeddingBag` layers, which *ONNX* does not support, are replaced automatically with an exportable equivalent at compile time.

The default `mode="dynamic_batch_size"` keeps the batch axis dynamic, so the same graph serves requests of any batch size. `max_seq_len` sets the maximum history length the graph accepts.

In [21]:
from replay.nn.compiled import compile

compiled = compile(twotower, schema, max_seq_len=MAX_SEQ_LEN)
compiled.input_names

[torch.onnx] Obtain model graph for `_TwoTowerQueryGraph([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `_TwoTowerQueryGraph([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅


['item_id', 'padding_mask']

In [22]:
manifest = compiled.manifest
manifest.model_type, manifest.exporter, manifest.max_seq_len, manifest.candidates_policy

('twotower_query', 'dynamo', 50, 'full_vocab')

In [23]:
manifest.input_shapes

{'item_id': ['batch', 'seq_len'], 'padding_mask': ['batch', 'seq_len']}

## Save and load the serving artifacts

`CompiledModel.save` writes the *ONNX* graph and its manifest; passing the `ItemEmbeddings` artifact stores the item matrix alongside it, so one directory holds everything the serving side needs.

`load` reads the artifact back and compiles it with *OpenVINO*. This path does not import PyTorch at all — on the serving side, `replay.nn.compiled` needs only `openvino`.

In [24]:
from pathlib import Path

from replay.nn.compiled import load, load_item_embeddings

artifact_dir = Path("serving_artifact")
compiled.save(artifact_dir, item_embeddings=item_embeddings)

sorted(path.name for path in artifact_dir.iterdir())

['item_embeddings.json', 'item_embeddings.npy', 'manifest.json', 'model.onnx']

In [25]:
compiled_serving = load(artifact_dir)
items_serving = load_item_embeddings(artifact_dir)
items_serving.matrix.shape

(3883, 64)

## Online inference with NumPy

On the serving side, user events arrive as plain dictionaries with raw (unencoded) values. `encode_inputs` maps them through the label-encoder mappings, applies truncation and padding, and returns NumPy arrays keyed by the graph input names.

We take one user from the validation set, convert their history back to raw item ids, and hold out the ground truth — the last event — for comparison.

In [26]:
sample = validation_events.iloc[0]
item_id_to_raw = {v: k for k, v in encoder.mapping["item_id"].items()}

user_events = [{"item_id": item_id_to_raw[item_id]} for item_id in sample["item_id"]]
ground_truth_raw = item_id_to_raw[sample["ground_truth"][0]]

len(user_events), user_events[:3], ground_truth_raw

(52, [{'item_id': 3186}, {'item_id': 1022}, {'item_id': 1270}], 48)

In [27]:
from replay.nn.compiled import encode_inputs

mappings = {"item_id": encoder.mapping["item_id"]}

inputs = encode_inputs(compiled_serving.manifest, [user_events], mappings)
{name: array.shape for name, array in inputs.items()}

{'item_id': (1, 50), 'padding_mask': (1, 50)}

`predict` runs the compiled query tower. The logits are a plain matrix product of the query embedding and the item matrix, so the top-10 recommendations are computed with NumPy alone.

In [28]:
query = compiled_serving.predict(inputs)

logits = query @ items_serving.matrix.T
top_idx = np.argsort(logits[0])[::-1][:10]

[item_id_to_raw[item] for item in top_idx]

[364, 595, 356, 1907, 588, 2080, 1022, 1029, 2081, 2078]

## Compare with PyTorch

Run the same inputs through the original PyTorch query tower and check that the compiled graph is numerically equivalent, then compare the latency of both paths on the same data.

In [29]:
feature_tensors = {name: torch.from_numpy(array) for name, array in inputs.items() if name != "padding_mask"}
padding_mask = torch.from_numpy(inputs["padding_mask"])

query_tower = twotower.body.query_tower
query_tower.eval()

with torch.no_grad():
    torch_query = query_tower(feature_tensors=feature_tensors, padding_mask=padding_mask)[:, -1, :].numpy()

np.allclose(torch_query, query, atol=1e-4, rtol=1e-3)

True

In [30]:
%%timeit -n 10 -r 10

with torch.no_grad():
    query_tower(feature_tensors=feature_tensors, padding_mask=padding_mask)[:, -1, :]

The slowest run took 11.15 times longer than the fastest. This could mean that an intermediate result is being cached.
3.01 ms ± 3.63 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [31]:
%%timeit -n 10 -r 10

compiled_serving.predict(inputs)

783 μs ± 255 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


## Summary

The serving side receives a single `serving_artifact/` directory — the *ONNX* graph, its manifest, and the item-embedding matrix — and answers requests with three NumPy-level calls:

1. `encode_inputs(manifest, events, mappings)` — raw event dicts to model inputs;
2. `compiled.predict(inputs)` — query embeddings from the *OpenVINO* graph;
3. a matrix product with the item matrix — candidate scores.

PyTorch is not imported on this path: `load` and `load_item_embeddings` need only `openvino`, and the config sidecar keeps the training checkpoint self-sufficient for future re-exports.